# `mutate` — Reference
`mutate` creates or overwrites columns using clean keyword argument syntax (`col="expr"` or `col=callable`). Each entry is evaluated in order via pandas `eval()` — a plain formula per column, or a lambda when needed.
### Calling Styles
1. **Keyword arguments** (most Pythonic): `.pt.mutate(bmi="body_mass_g / bill_length_mm ** 2", mass_kg="body_mass_g / 1000")`
2. **Dictionary unpacking** (for new column names with spaces): `.pt.mutate(**{"body mass kg": "body_mass_g / 1000"})`
3. **External file specification**: `.pt.mutate("@features.txt")`
---


In [1]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'src'))
if not os.path.exists(_src): _src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import numpy as np
import pytae as pt

penguins = pt.sample_data['penguins']

## A single derived column

In [2]:
# Body mass index style ratio — clean assignment formula via kwargs
(
    penguins
    .pt.mutate(bmi='body_mass_g / bill_length_mm ** 2')
    .pt.select('species', 'body_mass_g', 'bill_length_mm', 'bmi')
    .sample(10)
)


,species,body_mass_g,bill_length_mm,bmi
235,Gentoo,5850.0,49.3,2.406922
260,Gentoo,3950.0,42.7,2.166413
94,Adelie,3300.0,36.2,2.518238
336,Gentoo,4875.0,44.5,2.461810
42,Adelie,3100.0,36.0,2.391975
288,Gentoo,4700.0,43.5,2.483816
289,Gentoo,5550.0,50.7,2.159121
224,Gentoo,5400.0,47.6,2.383306
171,Chinstrap,4400.0,49.2,1.817701
164,Chinstrap,3700.0,47.0,1.674966


In [3]:
# Column names inside the expression must stay unquoted — quoting one turns it into
# a string literal, which will break arithmetic
try:
    penguins.pt.mutate(bmi="'body_mass_g' / 'bill_length_mm' ** 2")
except TypeError as exc:
    print(f"TypeError (as expected): {exc}")


TypeError (as expected): unsupported operand type(s) for ** or pow(): 'str' and 'int'


## Multiple entries in one call, and chaining a later entry off an earlier one
Entries are applied left to right, so a later expression can reference a column derived earlier in the *same* `mutate()` call.

In [4]:
# Two independent derived columns in one call via kwargs
(
    penguins
    .pt.mutate(heavy='body_mass_g > 4000', mass_kg='body_mass_g / 1000')
    .pt.select('species', 'body_mass_g', 'mass_kg', 'heavy')
    .sample(10)
)


,species,body_mass_g,mass_kg,heavy
72,Adelie,3550.0,3.55,False
42,Adelie,3100.0,3.10,False
53,Adelie,4050.0,4.05,True
40,Adelie,3150.0,3.15,False
185,Chinstrap,4100.0,4.10,True
107,Adelie,3900.0,3.90,False
247,Gentoo,5650.0,5.65,True
249,Gentoo,5550.0,5.55,True
276,Gentoo,4300.0,4.30,True
170,Chinstrap,3450.0,3.45,False


In [5]:
# mass_lb references mass_kg, derived by the keyword just before it
(
    penguins
    .pt.mutate(mass_kg='body_mass_g / 1000', mass_lb='mass_kg * 2.20462')
    .pt.select('species', 'body_mass_g', 'mass_kg', 'mass_lb')
    .sample(10)
)


,species,body_mass_g,mass_kg,mass_lb
263,Gentoo,4750.0,4.750,10.471945
91,Adelie,4300.0,4.300,9.479866
4,Adelie,3450.0,3.450,7.605939
51,Adelie,4300.0,4.300,9.479866
147,Adelie,3475.0,3.475,7.661054
92,Adelie,3400.0,3.400,7.495708
284,Gentoo,4700.0,4.700,10.361714
314,Gentoo,4850.0,4.850,10.692407
124,Adelie,3050.0,3.050,6.724091
182,Chinstrap,3200.0,3.200,7.054784


## String comparisons and local variables
String literals *inside* the expression (e.g. `'Adelie'`) need real quotes — column names stay bare. A variable from the calling scope can be referenced with an `@` prefix, same as pandas' own `eval()`/`query()`.


In [6]:
# String comparisons inside the expression formula
(
    penguins
    .pt.mutate(is_adelie="species == 'Adelie'")
    .pt.select('species', 'is_adelie')
    .sample(10)
)


,species,is_adelie
343,Gentoo,False
92,Adelie,True
295,Gentoo,False
304,Gentoo,False
47,Adelie,True
95,Adelie,True
217,Chinstrap,False
71,Adelie,True
86,Adelie,True
93,Adelie,True


In [7]:
# @-prefixed names resolve against the scope that called mutate(), not the module
# — works identically whether calling pt.mutate() or chaining .pt.mutate()
threshold = 4000

(
    penguins
    .pt.mutate(heavy='body_mass_g >= @threshold')
    .pt.select('species', 'body_mass_g', 'heavy')
    .sample(10)
)


,species,body_mass_g,heavy
148,Adelie,3450.0,False
78,Adelie,3550.0,False
52,Adelie,3450.0,False
4,Adelie,3450.0,False
255,Gentoo,5400.0,True
3,Adelie,NaN,False
336,Gentoo,4875.0,True
273,Gentoo,5000.0,True
229,Gentoo,5150.0,True
194,Chinstrap,3550.0,False


## Overwriting an existing column

In [8]:
# mutate() can overwrite a column in place, e.g. converting units
(
    penguins
    .pt.mutate(body_mass_g='body_mass_g / 1000')
    .pt.select('species', 'body_mass_g')
    .sample(10)
)


,species,body_mass_g
41,Adelie,3.900
33,Adelie,3.900
289,Gentoo,5.550
160,Chinstrap,4.150
134,Adelie,3.425
144,Adelie,3.000
216,Chinstrap,3.400
49,Adelie,4.150
28,Adelie,3.150
15,Adelie,3.700


## Column names with spaces — backtick quoting
`pandas.eval()` uses **backticks**, not the single/double quotes used elsewhere in pytae, to reference a column name containing a space.

In [9]:
import pandas as pd
spaced = pd.DataFrame({'body mass g': [3750, 4200], 'bill length mm': [39.1, 46.5]})

# Backticks protect column names that contain spaces
spaced.pt.mutate(bmi="`body mass g` / `bill length mm` ** 2")


,body mass g,bill length mm,bmi
0,3750,39.1,2.452888
1,4200,46.5,1.942421


## Real-world pipeline: mutate → filter → select
`mutate()` chains like any other pytae method — filter on a column you just derived with `qry()`.

In [10]:
(
    penguins
    .pt.mutate(bmi='body_mass_g / bill_length_mm ** 2')
    .pt.qry(bmi='> 2')
    .pt.select('species', 'island', 'body_mass_g', 'bill_length_mm', 'bmi')
    .sample(10)
)


,species,island,body_mass_g,bill_length_mm,bmi
223,Gentoo,Biscoe,5700.0,50.0,2.280000
1,Adelie,Torgersen,3800.0,39.5,2.435507
258,Gentoo,Biscoe,4350.0,44.0,2.246901
40,Adelie,Dream,3150.0,36.5,2.364421
303,Gentoo,Biscoe,5350.0,50.0,2.140000
282,Gentoo,Biscoe,4400.0,45.7,2.106785
34,Adelie,Dream,3325.0,36.4,2.509510
288,Gentoo,Biscoe,4700.0,43.5,2.483816
67,Adelie,Biscoe,4100.0,41.1,2.427170
46,Adelie,Dream,3425.0,41.1,2.027575


## Conditional column creation — `if_else()`, `case_when()`, `map()`
Plain `eval()` has no ternary/`where()` support, so `mutate()` provides three dplyr-style helpers as ordinary function calls, evaluated via `np.where()`/`np.select()`/`Series.map()`: `if_else(condition, true_value, false_value)`, `case_when((cond1, val1), (cond2, val2), ..., default)`, and `map(column, {key: value, ...}, default)`. A trailing bare argument to `case_when` is the catch-all default (like SQL ELSE). Because they are ordinary calls, they compose and chain with each other and with any pandas method. String outcomes need quotes; conditions are vectorized — prefer `and`/`or`/`not` (the bitwise `&`/`|`/`~` also work).

In [11]:
# if_else(condition, true_value, false_value) — like dplyr's if_else()
# true_value and false_value can be scalars or column names; conditions are vectorized
(
    penguins
    .pt.mutate(weight_class="if_else(body_mass_g > 4000, 'heavy', 'light')")
    .pt.select('species', 'body_mass_g', 'weight_class')
    .sample(10)
)


,species,body_mass_g,weight_class
331,Gentoo,5950.0,heavy
14,Adelie,4400.0,heavy
280,Gentoo,4200.0,heavy
234,Gentoo,4200.0,heavy
180,Chinstrap,3700.0,light
187,Chinstrap,3900.0,light
235,Gentoo,5850.0,heavy
250,Gentoo,5250.0,heavy
18,Adelie,3325.0,light
291,Gentoo,5000.0,heavy


In [12]:
# case_when supports tuples or flat pairs with default=
(
    penguins
    .pt.mutate(
        size_class="case_when(body_mass_g >= 4500, 'large', body_mass_g >= 3500, 'medium', default='small')"
    )
    .pt.select('species', 'body_mass_g', 'size_class')
    .sample(10)
)


,species,body_mass_g,size_class
124,Adelie,3050.0,small
297,Gentoo,6000.0,large
150,Adelie,3700.0,medium
86,Adelie,3800.0,medium
205,Chinstrap,4050.0,medium
6,Adelie,3625.0,medium
294,Gentoo,4700.0,large
289,Gentoo,5550.0,large
257,Gentoo,5250.0,large
192,Chinstrap,3950.0,medium


In [13]:
# map(column, {key: value, ...}, default) — recode a column through a dictionary lookup
# unmapped keys become default if given, else NaN
(
    penguins
    .pt.mutate(
        island_code="map(island, {'Torgersen': 'TOR', 'Biscoe': 'BIS'}, 'OTH')"
    )
    .pt.select('species', 'island', 'island_code')
    .sample(10)
)


,species,island,island_code
113,Adelie,Biscoe,BIS
11,Adelie,Torgersen,TOR
107,Adelie,Biscoe,BIS
272,Gentoo,Biscoe,BIS
83,Adelie,Torgersen,TOR
63,Adelie,Biscoe,BIS
290,Gentoo,Biscoe,BIS
248,Gentoo,Biscoe,BIS
225,Gentoo,Biscoe,BIS
241,Gentoo,Biscoe,BIS


## First non-null resolution — `coalesce()`

`coalesce(col1, col2, ..., default)` evaluates candidates left-to-right and returns the first non-null value per row, like SQL `COALESCE()` or `dplyr::coalesce()`:

In [14]:
contacts = pd.DataFrame({
    'mobile': [None, '555-1234', None],
    'home': ['555-5678', None, None],
    'work': [None, None, '555-9012'],
})

contacts.pt.mutate(preferred_contact="coalesce(mobile, home, work, 'N/A')")


,mobile,home,work,preferred_contact
0,NaN,555-5678,NaN,555-5678
1,555-1234,NaN,NaN,555-1234
2,NaN,NaN,555-9012,555-9012


## Combining expressions and callables in kwargs

`mutate()` cleanly mixes expression strings and python callables (e.g. lambdas) in keyword arguments:


In [15]:
(
    penguins
    .pt.mutate(
        bmi='body_mass_g / bill_length_mm ** 2',
        mass_kg='body_mass_g / 1000',
        is_heavy=lambda df: df['body_mass_g'] > 4000,
    )
    .pt.select('species', 'bmi', 'mass_kg', 'is_heavy')
    .head(5)
)


,species,bmi,mass_kg,is_heavy
0,Adelie,2.452888,3.75,False
1,Adelie,2.435507,3.80,False
2,Adelie,2.001121,3.25,False
3,Adelie,NaN,NaN,False
4,Adelie,2.561456,3.45,False


## Loading specs from an external file — `@specs.txt`
For complex feature engineering or shared pipelines across Python and the CLI, specs can be loaded from an external file. The file supports multiline expressions, `#` comments, empty lines, and column chaining (where later formulas reference columns defined earlier in the same file).
### File content of `penguin_features.txt`:
```text
# Engineering features for penguins
mass_kg = body_mass_g / 1000
# Convert kg to lbs (references mass_kg computed above)
mass_lb = mass_kg * 2.20462
```


In [16]:
# Write a small demo spec file with comments and assignment '=' syntax
spec_content = """# Engineering features for penguins
mass_kg = body_mass_g / 1000

# Convert kg to lbs
mass_lb = mass_kg * 2.20462
"""
with open("penguin_features.txt", "w") as f:
    f.write(spec_content)

# Load directly using @filename
result = penguins.pt.mutate("@penguin_features.txt")
out = (
    result
    .pt.select("species", "mass_kg", "mass_lb")
    .sample(5)
)

# Clean up demo file
import os
os.remove("penguin_features.txt")

out


,species,mass_kg,mass_lb
15,Adelie,3.70,8.157094
23,Adelie,3.95,8.708249
138,Adelie,3.40,7.495708
95,Adelie,4.30,9.479866
121,Adelie,3.50,7.716170


## Columns with spaces — SQL-style brackets `[col]`

SQL-style square brackets `[col a]` are supported alongside backticks, avoiding shell backtick substitution hazards:

In [17]:
spaced.pt.mutate(ratio='[body mass g] / [bill length mm]')


,body mass g,bill length mm,ratio
0,3750,39.1,95.907928
1,4200,46.5,90.322581


## Creating new columns with spaces — dictionary unpacking (`**{...}`)

Python keyword argument syntax requires valid identifier names (`key=value`), so bare names with spaces like `my col = "..."` cause a Python `SyntaxError`.

To create a new column whose name contains spaces, pass the column definition in a dictionary and unpack it using `**`:


In [18]:
# Create a new column with spaces using dictionary unpacking
(
    spaced
    .pt.mutate(**{'body mass ratio': '[body mass g] / [bill length mm]'})
)


,body mass g,bill length mm,body mass ratio
0,3750,39.1,95.907928
1,4200,46.5,90.322581


In [19]:
# Define multiple new columns with spaces at once
(
    penguins
    .pt.mutate(**{
        'body mass kg': 'body_mass_g / 1000',
        'bill ratio': 'bill_length_mm / bill_depth_mm',
    })
    .pt.select('species', 'body mass kg', 'bill ratio')
    .head()
)


,species,body mass kg,bill ratio
0,Adelie,3.75,2.090909
1,Adelie,3.80,2.270115
2,Adelie,3.25,2.238889
3,Adelie,NaN,NaN
4,Adelie,3.45,1.901554


Note: When loading specs from an external file (`@specs.txt`) or via the CLI (`-mutate`), names with spaces can be written directly on the left-hand side of `=` without dictionary unpacking:

```text
# inside an external spec file or CLI flag:
my bmi = [body mass g] / [bill length mm] ** 2
```
